# **Constraint Programming and Scheduling**

## **1. The problem: making decisions under constraints**

A **constraint satisfaction problem (CSP)** describes a situation in which we must choose values for several variables while obeying rules.

This notebook studies two versions of the same mathematical problem:

- **Map coloring:** assign a color to four regions so that neighboring regions have different colors.
- **Presentation scheduling:** assign a time slot to four students so that students with a conflict do not present together.

The important idea is the model, not the story. Once a problem is written as variables, domains, and constraints, the same search algorithm can solve both examples.

## **2. Modeling the map-coloring problem as a CSP**

A **constraint satisfaction problem** is defined in AIMA by three components:

$$CSP = \langle \mathcal{X}, \mathcal{D}, \mathcal{C} \rangle$$

- $\mathcal{X}$ is the set of variables.
- $\mathcal{D}$ is the collection of domains. Each variable $X_i$ has a domain $D_i$ of possible values.
- $\mathcal{C}$ is the set of constraints that specifies which combinations of values are allowed.

The calligraphic letters identify the three CSP components. Individual variables such as $X_i$ and individual domains such as $D_i$ remain ordinary italic mathematical symbols.

### **2.1 The four regions**

Imagine a map with four regions: **A, B, C, and D**.

```text
        A -------- B
         \        / \
          \      /   \
           C -------- D
```

The neighboring pairs are:

- A is a neighbor of B and C.
- B is a neighbor of A, C, and D.
- C is a neighbor of A, B, and D.
- D is a neighbor of B and C.

A neighbor is a region that shares a border with another region. Neighboring regions must have different colors. Non-neighboring regions may have the same color; for example, A and D may both be red.

### **2.2 Variables, domains, and constraints**

Using AIMA's notation:

- **Variables:** $\mathcal{X} = \{A, B, C, D\}$.
- **Domains:** $\mathcal{D} = \{D_A, D_B, D_C, D_D\}$, where $D_A = D_B = D_C = D_D = \{red, blue, green\}$.
- **Constraints:** $\mathcal{C}$ contains the neighboring-region inequalities:

$$A \ne B,\quad A \ne C,\quad B \ne C,\quad B \ne D,\quad C \ne D.$$

A solution is a **complete assignment** that gives one color to every variable and satisfies every constraint. For example:

$$A=red,\quad B=blue,\quad C=green,\quad D=red.$$

Before writing code, we separate the **generic solver** from this particular map. The solver will receive three inputs:

- `variables`: the variables in $\mathcal{X}$ that still need values.
- `domains`: the possible values in $\mathcal{D}$ for each variable.
- `neighbors`: the variables connected by a constraint in $\mathcal{C}$. In this problem, neighbors are regions sharing a border.

The next cell implements the generic backtracking algorithm. It does not contain the map data yet.

## **3. Backtracking search**

The map has several possible assignments, but we should not wait until the end to discover a conflict. **Backtracking search** builds a solution one variable at a time and prunes a partial assignment as soon as it violates a constraint.

For the map, the search can proceed as follows:

1. Choose an unassigned region, such as A.
2. Try one value from its domain, such as `red`.
3. Choose another region, such as B. Reject `red` because A and B are neighbors; try `blue` instead.
4. Choose C. Reject colors that conflict with A or B; keep `green`.
5. Choose D. Reject colors that conflict with B or C; keep `red`.
6. If a region has no valid color, undo the most recent choice and try the next available value.
7. When every region has a value, return the complete assignment.

The word **backtracking** describes step 6: the algorithm goes back to an earlier decision when the current partial assignment cannot be completed.

The next cell contains only the generic function. The map variables, domains, and neighbors are defined separately afterwards.

In [3]:
def solve_csp(variables, domains, neighbors):
    assignment = {}

    def backtrack():
        if len(assignment) == len(variables):
            return assignment.copy()

        variable = next(variable for variable in variables if variable not in assignment)
        for value in domains[variable]:
            if all(assignment.get(neighbor) != value for neighbor in neighbors[variable]):
                assignment[variable] = value
                result = backtrack()
                if result is not None:
                    return result
                del assignment[variable]
        return None

    return backtrack()

## **4. Define the map instance**

Now we create the concrete data for the four-region problem. This cell contains the variables, their domains, and the neighbor relation; it does not redefine the solver.

In [4]:
regions = ["A", "B", "C", "D"]

# Each region has the same domain: red, blue, or green.
colors = {
    "A": ["red", "blue", "green"],
    "B": ["red", "blue", "green"],
    "C": ["red", "blue", "green"],
    "D": ["red", "blue", "green"],
}

# These lists encode the border constraints.
neighboring_regions = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"],
}

coloring = solve_csp(regions, colors, neighboring_regions)
print("A valid coloring:", coloring)

A valid coloring: {'A': 'red', 'B': 'blue', 'C': 'green', 'D': 'red'}


## **5. Read the solver step by step**

The solver is now separate from the map. This makes the abstraction visible: the same function can later solve a scheduling problem.

- `variables` is AIMA's set $\mathcal{X}$.
- `domains[variable]` gives the individual domain $D_i$ from the collection $\mathcal{D}$.
- `neighbors[variable]` identifies variables connected by a binary constraint in $\mathcal{C}$.
- `assignment` is a partial assignment, such as `{A: red, B: blue}`.

Inside `backtrack()`:

1. `len(assignment) == len(variables)` checks whether the assignment is complete. If every variable has a value, the function returns a copy of the solution.
2. `next(...)` chooses an unassigned variable.
3. The `for` loop tries each value in that variable's domain.
4. `all(assignment.get(neighbor) != value ...)` tests the binary constraints. It compares only with neighbors already assigned.
5. A valid choice is added to `assignment`, and the function calls itself recursively.
6. If the recursive call returns `None`, the choice cannot lead to a complete solution. `del assignment[variable]` removes it so the next value can be tried.

For the map, the result represents the function:

$$A=red,\quad B=blue,\quad C=green,\quad D=red.$$

The algorithm is correct when every neighboring pair receives different values. A and D may have the same color because they are not neighbors.

### **Check your understanding**

Use the map example, not an abstract definition, to answer these questions:

1. What are the four variables, and what is the domain of each variable?
2. Why is `{A: red, B: red}` rejected immediately?
3. Why is `{A: red, D: red}` allowed?
4. If A is `red` and B is `blue`, which colors remain possible for C?
5. What does the search return if no color is valid for an unassigned region?

**Instructor prompt:** ask students to write one constraint as an inequality, then explain why that inequality is checked before the assignment is complete.

## **6. Lab: schedule student presentations**

Now apply exactly the same CSP model to a timetable.

Four students (`Ada`, `Ben`, `Cy`, `Dee`) must present in one of three time slots (`9:00`, `10:00`, `11:00`). Students who are working together must present at different times. The conflict pairs are `Ada–Ben`, `Ben–Cy`, and `Cy–Dee`.

### **6.1 Translate the story into AIMA's CSP notation**

- **Variables:** $\mathcal{X} = \{Ada, Ben, Cy, Dee\}$.
- **Domains:** $\mathcal{D} = \{D_{Ada}, D_{Ben}, D_{Cy}, D_{Dee}\}$, where each domain is $\{9:00, 10:00, 11:00\}$.
- **Constraints:** $\mathcal{C}$ contains $Ada \ne Ben$, $Ben \ne Cy$, and $Cy \ne Dee$.

The `conflicts` dictionary will store the neighbor relation. For example, `conflicts["Ben"]` contains Ada and Cy because Ben cannot share a slot with either of them.

A partial assignment such as `{Ada: 9:00, Ben: 9:00}` is invalid because Ada and Ben conflict. The partial assignment `{Ada: 9:00, Dee: 9:00}` is allowed because they are not neighbors. This is exactly the same reasoning as with regions A and D in the map.

### **6.2 Tasks**

1. Read the variables, domains, and conflict relation in the next code cell.
2. Complete `is_consistent` so it returns `False` when an already scheduled neighbor has the same slot.
3. Complete the recursive `search` function. Return a complete assignment, or `None` if every possible choice violates a constraint.
4. Run the supplied checks and add one extra conflict pair of your choice. Does the schedule change?
5. In two sentences, explain where pruning happens and why checking partial assignments is useful.

In [ ]:
students = ["Ada", "Ben", "Cy", "Dee"]
slots = ["9:00", "10:00", "11:00"]
domains = {student: slots[:] for student in students}
conflicts = {
    "Ada": ["Ben"],
    "Ben": ["Ada", "Cy"],
    "Cy": ["Ben", "Dee"],
    "Dee": ["Cy"],
}

def is_consistent(student, slot, assignment):
    # TODO: return False if an already scheduled partner has this slot.
    raise NotImplementedError

def search(assignment):
    # TODO: return a complete solution, or None if all choices fail.
    # Hint: use recursion and remove a tentative assignment after failure.
    raise NotImplementedError

schedule = search({})
print(schedule)

In [ ]:
# Run after completing the starter code.
assert schedule is not None
assert set(schedule) == set(students)
assert all(schedule[a] != schedule[b] for a, partners in conflicts.items() for b in partners)
print("All scheduling constraints are satisfied.")

## **7. Exit ticket and further reading**

Submit your notebook with the completed code, passing checks, the schedule you found, and your short explanation of pruning. In your explanation, name the variables, domains, and at least two constraints from the scheduling problem.

**Extension:** add a fourth slot, or add a preference (such as Ada preferring 10:00) and discuss how optimization differs from finding any feasible schedule. A preference is usually a **soft constraint**: it is desirable, but it may be violated when necessary. A conflict between two students is a **hard constraint**: it must never be violated.

### **AIMA reference**

For the formal definition of constraint satisfaction problems, variables, domains, constraints, backtracking search, and heuristics, read:

- Russell, S. and Norvig, P., *Artificial Intelligence: A Modern Approach*, 4th ed., Chapter 6, **Constraint Satisfaction Problems**.
- In particular, see Sections **6.1 (Defining Constraint Satisfaction Problems)** and **6.3 (Backtracking Search for CSPs)**.

The map-coloring problem in this notebook is the standard graph-coloring CSP discussed in that chapter; the presentation timetable is the same mathematical structure with different variable and domain names.